In [1]:
import json
import os

In [2]:
DATASET_FOLDER = "Datasets"

In [3]:
info_list = []

for item in os.scandir(DATASET_FOLDER):
    if item.is_dir():
        # is folder
        folder_name = item.name
        
        # check if last token is int
        if folder_name.split("_")[0] == "tabular" or folder_name.split("_")[0] == "image":
            # site folder
            # print("Valid folder:", item.path)

            # Look for JSON files inside
            for sub in os.scandir(item.path):
                if sub.is_file() and sub.name.startswith(folder_name) and sub.name.endswith("stats.json"):
                    json_file = sub.path
                    # print("  Found JSON:", json_file)

                    # Load it
                    with open(json_file, "r") as f:
                        data = json.load(f)
                    info_list.append(data)

info_list

[{'site_name': 'tabular_1',
  'n_samples': 229712,
  'feature_means': {'Age': 8.085981576931113,
   'Sex': 0.4391411854844327,
   'BMI': 28.68571080309257,
   'GenHlth': 2.6012049871142997,
   'HighBP': 0.4543776554990597,
   'DiffWalk': 0.1855584383924218,
   'HighChol': 0.441696559169743,
   'HeartDiseaseorAttack': 0.10321620115623041},
  'feature_vars': {'Age': 9.5726421687767,
   'Sex': 0.2462962046957598,
   'BMI': 46.064766216567136,
   'GenHlth': 1.1337465454985667,
   'HighBP': 0.24791860168223753,
   'DiffWalk': 0.15112650433378755,
   'HighChol': 0.24660070878735263,
   'HeartDiseaseorAttack': 0.09256261697510698}}]

In [4]:
from collections import defaultdict
import numpy as np

site_stats = info_list

In [5]:
# ---- define the global feature template ----
feature_template = [
    "Age", "Sex", "BMI", "GenHlth",
    "HighBP", "DiffWalk", "HighChol", "HeartDiseaseorAttack"
]

In [6]:
# ---- compute global means ----
sum_mu = defaultdict(float)
sum_n  = defaultdict(int)

for site in site_stats:
    n = int(site["n_samples"])
    for f in feature_template:
        if f in site["feature_means"]:
            sum_mu[f] += n * site["feature_means"][f]
            sum_n[f]  += n

global_mean = {
    f: (sum_mu[f] / sum_n[f]) if sum_n[f] > 0 else 0.0
    for f in feature_template
}

# ---- compute global variances ----
sum_var = defaultdict(float)

for site in site_stats:
    n = int(site["n_samples"])
    for f in feature_template:
        if f in site["feature_means"]:
            mu_i = site["feature_means"][f]
            var_i = site["feature_vars"][f]
            sum_var[f] += n * (var_i + (mu_i - global_mean[f])**2)

global_var = {
    f: (sum_var[f] / sum_n[f]) if sum_n[f] > 0 else 0.0
    for f in feature_template
}

# ---- compute std ----
global_std = {f: float(np.sqrt(v)) for f, v in global_var.items()}

# ---- final dict ----
global_stats = {
    "global_feature_mean": global_mean,
    "global_feature_std": global_std
}

In [7]:
global_stats

{'global_feature_mean': {'Age': 8.085981576931113,
  'Sex': 0.4391411854844327,
  'BMI': 28.68571080309257,
  'GenHlth': 2.6012049871142997,
  'HighBP': 0.4543776554990597,
  'DiffWalk': 0.1855584383924218,
  'HighChol': 0.441696559169743,
  'HeartDiseaseorAttack': 0.10321620115623041},
 'global_feature_std': {'Age': 3.0939686761143363,
  'Sex': 0.4962823840272389,
  'BMI': 6.787102932515989,
  'GenHlth': 1.0647753497797396,
  'HighBP': 0.49791425133474293,
  'DiffWalk': 0.388749925188144,
  'HighChol': 0.4965890743737246,
  'HeartDiseaseorAttack': 0.3042410507724212}}

In [8]:
# save to file
with open("global_tabular_stats.json", "w", encoding="utf-8") as f:
    json.dump(global_stats, f, indent=2, ensure_ascii=False)